In [7]:
import sys
!{sys.executable} -m pip install xlrd

  Using cached xlrd-2.0.2-py2.py3-none-any.whl.metadata (3.5 kB)
Using cached xlrd-2.0.2-py2.py3-none-any.whl (96 kB)


In [8]:
import pandas as pd

# --- Load both datasets ---
csv_path = "Custom_Crops_yield_Historical_Dataset.csv"
xls_path = "icrisat.xls"
df_historical = pd.read_csv(csv_path)
df_icrisat = pd.read_excel(xls_path)

# --- Basic shape and structure ---
print("=== HISTORICAL CROP YIELD DATASET ===")
print("Shape:", df_historical.shape)
print("\nColumns:\n", df_historical.columns.tolist())
print("\nDtypes:\n", df_historical.dtypes)
print("\nFirst 5 rows:\n", df_historical.head())
print("\nMissing values:\n", df_historical.isnull().sum())

print("\n\n=== ICRISAT DISTRICT-LEVEL DATASET ===")
print("Shape:", df_icrisat.shape)
print("\nColumns:\n", df_icrisat.columns.tolist())
print("\nDtypes:\n", df_icrisat.dtypes)
print("\nFirst 5 rows:\n", df_icrisat.head())
print("\nMissing values:\n", df_icrisat.isnull().sum())

# --- Check for common keys to merge on later (district, state, year) ---
print("\n\n=== POTENTIAL MERGE KEYS ===")
for col in df_historical.columns:
    if any(key in col.lower() for key in ['district', 'state', 'year']):
        print("Historical dataset has:", col)

for col in df_icrisat.columns:
    if any(key in col.lower() for key in ['district', 'state', 'year']):
        print("ICRISAT dataset has:", col)

=== HISTORICAL CROP YIELD DATASET ===
Shape: (50765, 20)

Columns:
 ['Dist Code', 'Year', 'State Code', 'State Name', 'Dist Name', 'Crop', 'Area_ha', 'Yield_kg_per_ha', 'N_req_kg_per_ha', 'P_req_kg_per_ha', 'K_req_kg_per_ha', 'Total_N_kg', 'Total_P_kg', 'Total_K_kg', 'Temperature_C', 'Humidity_%', 'pH', 'Rainfall_mm', 'Wind_Speed_m_s', 'Solar_Radiation_MJ_m2_day']

Dtypes:
 Dist Code                      int64
Year                           int64
State Code                     int64
State Name                       str
Dist Name                        str
Crop                             str
Area_ha                      float64
Yield_kg_per_ha              float64
N_req_kg_per_ha              float64
P_req_kg_per_ha              float64
K_req_kg_per_ha              float64
Total_N_kg                   float64
Total_P_kg                   float64
Total_K_kg                   float64
Temperature_C                  int64
Humidity_%                     int64
pH                           fl

In [15]:
# Cell 1
print(df_icrisat.shape)
print(df_icrisat.columns.tolist(),"\n")

(12803, 107)
['Dist Code', 'Year', 'State Code', 'State Name', 'Dist Name', 'RICE AREA (1000 ha)', 'RICE PRODUCTION (1000 tons)', 'RICE YIELD (Kg per ha)', 'PEARL MILLET AREA (1000 ha)', 'PEARL MILLET PRODUCTION (1000 tons)', 'PEARL MILLET YIELD (Kg per ha)', 'CHICKPEA AREA (1000 ha)', 'CHICKPEA PRODUCTION (1000 tons)', 'CHICKPEA YIELD (Kg per ha)', 'GROUNDNUT AREA (1000 ha)', 'GROUNDNUT PRODUCTION (1000 tons)', 'GROUNDNUT YIELD (Kg per ha)', 'SUGARCANE AREA (1000 ha)', 'SUGARCANE PRODUCTION (1000 tons)', 'SUGARCANE YIELD (Kg per ha)', 'GROSS CROPPED AREA (1000 ha)', 'NITROGEN CONSUMPTION (tons)', 'PHOSPHATE CONSUMPTION (tons)', 'POTASH CONSUMPTION (tons)', 'TOTAL FERTILISER CONSUMPTION (tons)', 'TOTAL AGRICULTURAL LABOUR POPULATION (1000 Number)', 'GROSS IRRIGATED AREA (1000 ha)', 'JANUARY MAXIMUM TEMPERATURE (Centigrate)', 'FEBRUARY MAXIMUM TEMPERATURE (Centigrate)', 'MARCH MAXIMUM TEMPERATURE (Centigrate)', 'APRIL MAXIMUM TEMPERATURE (Centigrate)', 'MAY MAXIMUM TEMPERATURE (Centigra

In [10]:
# Cell 2
print(df_icrisat.dtypes)

Dist Code                                        int64
Year                                             int64
State Code                                       int64
State Name                                         str
Dist Name                                          str
                                                ...   
DEC WINDSPEED (Meter per second)               float64
Winter JAN-FEB WINDSPEED (Meter per second)    float64
Summer MAR-MAY WINDSPEED (Meter per second)    float64
Rainy JUN-SEP WINDSPEED (Meter per second)     float64
Autumn OCT-DEC WINDSPEED (Meter per second)    float64
Length: 107, dtype: object


In [11]:
# Cell 3
print(df_icrisat.isnull().sum())

Dist Code                                        0
Year                                             0
State Code                                       0
State Name                                       0
Dist Name                                        0
                                              ... 
DEC WINDSPEED (Meter per second)               697
Winter JAN-FEB WINDSPEED (Meter per second)    697
Summer MAR-MAY WINDSPEED (Meter per second)    697
Rainy JUN-SEP WINDSPEED (Meter per second)     697
Autumn OCT-DEC WINDSPEED (Meter per second)    697
Length: 107, dtype: int64


In [16]:
hist_districts = set(df_historical['Dist Name'].str.strip().str.upper().unique())
icrisat_districts = set(df_icrisat['Dist Name'].str.strip().str.upper().unique())  # adjust column name once known

print("In historical but not ICRISAT:", hist_districts - icrisat_districts)
print("In ICRISAT but not historical:", icrisat_districts - hist_districts)

In historical but not ICRISAT: {'24 PARGANAS', 'WEST DINAJPUR', 'BIJAPUR / VIJAYAPURA', 'MIDNAPUR', 'ANANTHAPUR', 'SINGHBHUM', 'KHANDWA / EAST NIMAR', 'BOMBAY', 'BANGALORE', 'GULBARGA / KALABURAGI', 'SEONI / SHIVANI', 'KODAGU / COORG', 'MAHENDRAGARH / NARNAUL', 'KHARGONE / WEST NIMAR', 'SHAHABAD (NOW PART OF BHOJPUR DISTRICT)', 'CHENGALPATTU MGR / KANCHIPURAM', 'KANPUR', 'VADODARA / BARODA', 'NORTH CACHAR HIL / DIMA HASAO', 'CHAMPARAN', 'ROOPNAGAR / ROPAR', 'HOWRAH'}
In ICRISAT but not historical: {'FARIDABAD', 'MADHEPURA', 'MAU', 'UDHAM SINGH NAGAR', 'KANNAUJ', 'MADHUBANI', 'SONEPUR', '24-PARAGANAS SOUTH', 'KODAGU', 'KAWARDHA', 'NANDURBAR', 'MUMBAI SUB', 'THIRUPPUR', 'SAHEBGANJ', 'MAHISAGAR', 'FAZILKA', 'UTTAR DINAJPUR', 'DINDIGUL ANNA', 'SHEIKAPURA', 'MUMBAI CITY', 'KISHANGANJ', 'ANANTAPUR', 'VIRUDHUNAGAR / KAMARAJAR', 'SAMBHAL', 'BHOJPUR', 'CHAMPARAN (EAST)', 'DHARMAPURI', 'KORBA', 'CHANDAULI', 'DHUBRI', 'PANIPAT', 'KAITHAL', 'JASHPUR', 'BALOD', 'SINGHBHUM WEST', 'NARAYANPUR', 'CHIR

In [17]:
print("Historical years:", df_historical['Year'].min(), "-", df_historical['Year'].max())
print("ICRISAT years:", df_icrisat['Year'].min(), "-", df_icrisat['Year'].max())

Historical years: 1966 - 2017
ICRISAT years: 1990 - 2015


In [18]:
"""
District Name Standardization & Merge Script
----------------------------------------------
Purpose: Reconcile district-name mismatches between the Historical Crop
Yield dataset and the ICRISAT District-Level dataset before merging.

Strategy:
1. Standardize both datasets' district names to uppercase, stripped of spaces.
2. Apply a manual mapping dictionary for known renames/splits (built from the
   "X / Y" pattern already present in the historical dataset, plus obvious
   spelling variants).
3. For districts that were split into multiple new districts over time
   (e.g. Bangalore -> Bangalore Urban + Bangalore Rural), aggregate the ICRISAT
   split-district rows back to the parent name so historical-era rows can match.
4. Report and drop rows that remain unmatched (log the count/percentage —
   this becomes a documented limitation in your paper, which is normal).
"""

import pandas as pd

# --- Load datasets ---
df_historical = pd.read_csv("Custom_Crops_yield_Historical_Dataset.csv")
df_icrisat = pd.read_excel("icrisat.xls")

# --- Step 1: Standardize casing/whitespace ---
df_historical["Dist Name"] = df_historical["Dist Name"].str.strip().str.upper()
df_icrisat["Dist Name"] = df_icrisat["Dist Name"].str.strip().str.upper()

df_historical["State Name"] = df_historical["State Name"].str.strip().str.upper()
df_icrisat["State Name"] = df_icrisat["State Name"].str.strip().str.upper()

# --- Step 2: Handle "OLD / NEW" naming pattern already present in historical data ---
# These rows literally encode old_name -> new_name. We split them and create
# two mapping directions so either dataset's naming can be normalized to a
# single canonical name (we'll standardize to the "new" / post-rename name).

rename_pairs = {
    "BIJAPUR / VIJAYAPURA": "VIJAYAPURA",
    "GULBARGA / KALABURAGI": "KALABURAGI",
    "KODAGU / COORG": "KODAGU",
    "KHANDWA / EAST NIMAR": "KHANDWA",
    "KHARGONE / WEST NIMAR": "KHARGONE",
    "MAHENDRAGARH / NARNAUL": "MAHENDRAGARH",
    "CHENGALPATTU MGR / KANCHIPURAM": "KANCHIPURAM",
    "VADODARA / BARODA": "VADODARA",
    "NORTH CACHAR HIL / DIMA HASAO": "DIMA HASAO",
    "ROOPNAGAR / ROPAR": "ROOPNAGAR",
    "SEONI / SHIVANI": "SEONI",
    "ANANTHAPUR": "ANANTAPUR",          # spelling variant
    "GULBARGA": "KALABURAGI",           # standalone old-name rows in ICRISAT
    "BIJAPUR": "VIJAYAPURA",            # NOTE: verify manually — Karnataka's
                                          # Vijayapura vs Chhattisgarh's Bijapur
                                          # can collide; check State Name too.
    "24 PARGANAS": "24-PARAGANAS",       # will aggregate split N/S separately
    "WEST DINAJPUR": "DAKSHIN DINAJPUR", # approximate historical parent
    "MIDNAPUR": "MIDNAPORE",             # will aggregate East/West separately
    "SHAHABAD (NOW PART OF BHOJPUR DISTRICT)": "BHOJPUR",
    "CHAMPARAN": "CHAMPARAN",            # will aggregate East/West separately
    "KANPUR": "KANPUR NAGAR",            # approximate — verify
    "BOMBAY": "MUMBAI CITY",             # approximate — verify
    "BANGALORE": "BANGALORE",            # will aggregate Urban/Rural separately
    "HOWRAH": "HOWRAH",                  # present in both, likely just needs upper()
    "KODAGU / COORG": "KODAGU",
}

def apply_rename(name):
    return rename_pairs.get(name, name)

df_historical["Dist Name Clean"] = df_historical["Dist Name"].apply(apply_rename)
df_icrisat["Dist Name Clean"] = df_icrisat["Dist Name"].apply(apply_rename)

# --- Step 3: Aggregate split districts in ICRISAT back to a parent name ---
# These parent districts were later divided into multiple new districts.
# We sum/average their split-district data back under the historical parent
# name so it can merge against pre-split historical rows.
# NOTE: Area/Production columns should be SUMMED; Yield/Temperature/Rainfall/
# etc. (rate or intensity measures) should be AVERAGED, not summed.

split_district_map = {
    "BANGALORE (URBAN)": "BANGALORE",
    "BANGALORE (RURAL)": "BANGALORE",
    "RAMANAGARAM": "BANGALORE",  # Ramanagara was carved from Bangalore Rural
    "24-PARAGANAS NORTH": "24-PARAGANAS",
    "24-PARAGANAS SOUTH": "24-PARAGANAS",
    "WEST MIDNAPORE": "MIDNAPORE",
    "EAST MIDNAPORE / PURBA MIDNAPORE": "MIDNAPORE",
    "CHAMPARAN (EAST)": "CHAMPARAN",
    "CHAMPARAN (WEST)": "CHAMPARAN",
    "SINGHBHUM EAST": "SINGHBHUM",
    "SINGHBHUM WEST": "SINGHBHUM",
    "DAKSHIN DINAJPUR": "WEST DINAJPUR",
    "UTTAR DINAJPUR": "WEST DINAJPUR",
}

df_icrisat["Dist Name Clean"] = df_icrisat["Dist Name Clean"].replace(split_district_map)

# Identify which columns are "sum-able" (area/production/count) vs
# "average-able" (yield/temperature/rainfall/rates) for aggregation.
sum_keywords = ["AREA", "PRODUCTION", "CONSUMPTION", "POPULATION"]
sum_cols = [c for c in df_icrisat.columns if any(k in c.upper() for k in sum_keywords)]
avg_cols = [c for c in df_icrisat.columns
            if c not in sum_cols
            and c not in ["Dist Code", "Year", "State Code", "State Name",
                           "Dist Name", "Dist Name Clean"]]

agg_dict = {col: "sum" for col in sum_cols}
agg_dict.update({col: "mean" for col in avg_cols})

df_icrisat_agg = (
    df_icrisat
    .groupby(["Dist Name Clean", "State Name", "Year"], as_index=False)
    .agg(agg_dict)
)

# --- Step 4: Check remaining mismatches after cleaning ---
hist_districts = set(df_historical["Dist Name Clean"].unique())
icrisat_districts = set(df_icrisat_agg["Dist Name Clean"].unique())

still_unmatched_hist = hist_districts - icrisat_districts
still_unmatched_icrisat = icrisat_districts - hist_districts

print(f"Historical unique districts: {len(hist_districts)}")
print(f"ICRISAT unique districts (after aggregation): {len(icrisat_districts)}")
print(f"\nStill unmatched (historical side): {len(still_unmatched_hist)}")
print(still_unmatched_hist)
print(f"\nStill unmatched (ICRISAT side): {len(still_unmatched_icrisat)}")
print(still_unmatched_icrisat)

# --- Step 5: Merge on cleaned district name + state + year ---
merged = pd.merge(
    df_historical,
    df_icrisat_agg,
    left_on=["Dist Name Clean", "State Name", "Year"],
    right_on=["Dist Name Clean", "State Name", "Year"],
    how="inner",
    suffixes=("_hist", "_icrisat")
)

print(f"\nMerged dataset shape: {merged.shape}")
print(f"Original historical rows: {df_historical.shape[0]}")
print(f"Rows retained after merge: {merged.shape[0]} "
      f"({merged.shape[0] / df_historical.shape[0] * 100:.1f}% of historical data)")

# --- Step 6: Save outputs ---
merged.to_csv("merged_dataset.csv", index=False)

unmatched_report = pd.DataFrame({
    "unmatched_district": list(still_unmatched_hist) + list(still_unmatched_icrisat),
    "source": (["historical"] * len(still_unmatched_hist)
               + ["icrisat"] * len(still_unmatched_icrisat))
})
unmatched_report.to_csv("unmatched_districts_report.csv", index=False)

print("\nSaved: merged_dataset.csv")
print("Saved: unmatched_districts_report.csv (for documentation in your paper)")

C:\Users\shrey\AppData\Local\Temp\ipykernel_22616\2535539951.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_icrisat["Dist Name Clean"] = df_icrisat["Dist Name"].apply(apply_rename)
C:\Users\shrey\AppData\Local\Temp\ipykernel_22616\2535539951.py:112: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .agg(agg_dict)


Historical unique districts: 311
ICRISAT unique districts (after aggregation): 548

Still unmatched (historical side): 4
{'DAKSHIN DINAJPUR', 'DIMA HASAO', 'KANCHIPURAM', 'HOWRAH'}

Still unmatched (ICRISAT side): 241
{'FARIDABAD', 'MADHEPURA', 'MAU', 'KANNAUJ', 'UDHAM SINGH NAGAR', 'MADHUBANI', 'SONEPUR', 'KAWARDHA', 'NANDURBAR', 'MUMBAI SUB', 'THIRUPPUR', 'SAHEBGANJ', 'MAHISAGAR', 'FAZILKA', 'DINDIGUL ANNA', 'SHEIKAPURA', 'KISHANGANJ', 'VIRUDHUNAGAR / KAMARAJAR', 'SAMBHAL', 'DHARMAPURI', 'KORBA', 'CHANDAULI', 'DHUBRI', 'PANIPAT', 'KAITHAL', 'JASHPUR', 'BALOD', 'NARAYANPUR', 'CHIRANG', 'UMARIA', 'THENI', 'NEEMUCH', 'G.B.NAGAR', 'JHAJJAR', 'BEMETRA', 'ALIPURDUAR', 'BAGESHWAR', 'SONBHADRA', 'JHARSUGUDA', 'DEVGHAR / DEOGARH', 'VAISHALI', 'ARIYALUR', 'DAUSA', 'HARDA', 'BOKARO', 'NARMADA', 'KENDRAPARA', 'MALKANGIRI', 'SIVAGANGAI / PASUMPON', 'PRAKASAM', 'ROHTAS', 'SUKMA', 'RANGAREDDY', 'TINSUKIA', 'GUMLA', 'NAVSARI', 'ANAND', 'BARPETA', 'JAJAPUR', 'GADCHIROLI', 'RAYAGADA', 'SONEPAT', 'PATA

In [19]:
print(f"Historical unique districts: {len(hist_districts)}")
print(f"ICRISAT unique districts (after aggregation): {len(icrisat_districts)}")
print(f"Still unmatched (historical side): {len(still_unmatched_hist)}")
print(f"Still unmatched (ICRISAT side): {len(still_unmatched_icrisat)}")
print(f"Merged dataset shape: {merged.shape}")
print(f"Rows retained: {merged.shape[0] / df_historical.shape[0] * 100:.1f}%")

Historical unique districts: 311
ICRISAT unique districts (after aggregation): 548
Still unmatched (historical side): 4
Still unmatched (ICRISAT side): 241
Merged dataset shape: (25210, 123)
Rows retained: 49.7%


In [20]:
import pandas as pd
merged = pd.read_csv("merged_dataset.csv")

missing = merged.isnull().sum()
missing_pct = (missing / len(merged)) * 100
missing_summary = pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_%", ascending=False)
print(missing_summary)

                                             missing_count  missing_%
Autumn OCT-DEC WINDSPEED (Meter per second)           1738   6.894090
JAN WINDSPEED (Meter per second)                      1738   6.894090
Rainy JUN-SEP WINDSPEED (Meter per second)            1738   6.894090
Summer MAR-MAY WINDSPEED (Meter per second)           1738   6.894090
Winter JAN-FEB WINDSPEED (Meter per second)           1738   6.894090
...                                                    ...        ...
MAY MINIMUM TEMPERATURE (Centigrate)                   403   1.598572
SUGARCANE YIELD (Kg per ha)                            364   1.443871
GROUNDNUT YIELD (Kg per ha)                            199   0.789369
CHICKPEA YIELD (Kg per ha)                              45   0.178501
RICE YIELD (Kg per ha)                                  18   0.071400

[85 rows x 2 columns]


In [21]:
# Example: fill numeric columns with district-level mean, fallback to state-level mean
numeric_cols = merged.select_dtypes(include='number').columns

for col in numeric_cols:
    merged[col] = merged.groupby('Dist Name Clean')[col].transform(lambda x: x.fillna(x.mean()))
    merged[col] = merged.groupby('State Name')[col].transform(lambda x: x.fillna(x.mean()))

# Any still-missing after that (entire district/state missing that column) — drop
merged = merged.dropna(subset=numeric_cols, thresh=int(0.8 * len(numeric_cols)))

In [22]:
print("Final shape:", merged.shape)
print("Remaining missing values:", merged.isnull().sum().sum())
merged.to_csv("merged_dataset_cleaned.csv", index=False)

Final shape: (25210, 123)
Remaining missing values: 0


In [23]:
# Check if missing yield correlates with missing/zero area for that crop
print(merged[merged['RICE YIELD (Kg per ha)'].isnull()][['RICE AREA (1000 ha)']].describe())

       RICE AREA (1000 ha)
count                  0.0
mean                   NaN
std                    NaN
min                    NaN
25%                    NaN
50%                    NaN
75%                    NaN
max                    NaN


In [24]:
merged.to_csv("merged_dataset_cleaned.csv", index=False)

In [25]:
def row_to_text(row):
    return (f"In {row['Dist Name Clean']} district, {row['State Name']}, "
            f"in {row['Year']}, rice yield was {row['RICE YIELD (Kg per ha)']} kg/ha, "
            f"groundnut yield was {row['GROUNDNUT YIELD (Kg per ha)']} kg/ha, "
            f"with average rainfall of {row['JULY PERCIPITATION (Millimeters)']} mm in July "
            f"and total fertiliser consumption of {row['TOTAL FERTILISER CONSUMPTION (tons)']} tons.")

merged['rag_text'] = merged.apply(row_to_text, axis=1)
merged[['Dist Name Clean', 'State Name', 'Year', 'rag_text']].to_csv("rag_chunks.csv", index=False)

C:\Users\shrey\AppData\Local\Temp\ipykernel_22616\2886064482.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged['rag_text'] = merged.apply(row_to_text, axis=1)


building the actual RAG text chunks

In [26]:
def row_to_text(row):
    return (f"In {row['Dist Name Clean']} district, {row['State Name']}, "
            f"in {row['Year']}, rice yield was {row['RICE YIELD (Kg per ha)']} kg/ha, "
            f"groundnut yield was {row['GROUNDNUT YIELD (Kg per ha)']} kg/ha, "
            f"chickpea yield was {row['CHICKPEA YIELD (Kg per ha)']} kg/ha, "
            f"with average July rainfall of {row['JULY PERCIPITATION (Millimeters)']} mm "
            f"and total fertiliser consumption of {row['TOTAL FERTILISER CONSUMPTION (tons)']} tons.")

merged = merged.copy()
merged['rag_text'] = merged.apply(row_to_text, axis=1)

# Save the chunks for the RAG pipeline
rag_output = merged[['Dist Name Clean', 'State Name', 'Year', 'rag_text']]
rag_output.to_csv("rag_chunks.csv", index=False)

print(rag_output.head(3))
print(f"\nTotal chunks created: {len(rag_output)}")

  Dist Name Clean    State Name  Year  \
0            DURG  CHHATTISGARH  1990   
1            DURG  CHHATTISGARH  1990   
2            DURG  CHHATTISGARH  1990   

                                            rag_text  
0  In DURG district, CHHATTISGARH, in 1990, rice ...  
1  In DURG district, CHHATTISGARH, in 1990, rice ...  
2  In DURG district, CHHATTISGARH, in 1990, rice ...  

Total chunks created: 25210


In [27]:
#remove duplicateeeee
# Drop exact duplicate chunks (keeps first occurrence)
rag_output_deduped = rag_output.drop_duplicates(subset=['Dist Name Clean', 'State Name', 'Year'])
print(f"Before dedup: {len(rag_output)}")
print(f"After dedup: {len(rag_output_deduped)}")

rag_output_deduped.to_csv("rag_chunks_final.csv", index=False)

Before dedup: 25210
After dedup: 7943


In [28]:
#cant throw away so much data
def row_to_text(row):
    return (f"In {row['Dist Name Clean']} district, {row['State Name']}, in {row['Year']}, "
            f"{row['Crop']} had a yield of {row['Yield_kg_per_ha']} kg/ha "
            f"grown over {row['Area_ha']} hectares, requiring {row['N_req_kg_per_ha']} kg/ha nitrogen, "
            f"{row['P_req_kg_per_ha']} kg/ha phosphorus, and {row['K_req_kg_per_ha']} kg/ha potassium. "
            f"Regional conditions included {row['Rainfall_mm']} mm rainfall, "
            f"{row['Temperature_C']}°C average temperature, and {row['Humidity_%']}% humidity. "
            f"District-level fertiliser consumption that year was "
            f"{row['TOTAL FERTILISER CONSUMPTION (tons)']} tons.")

merged = merged.copy()
merged['rag_text'] = merged.apply(row_to_text, axis=1)

# Now check for TRUE duplicates (should be near-zero, since crop makes each row distinct)
print(f"Total rows: {len(merged)}")
print(f"Unique rag_text: {merged['rag_text'].nunique()}")

rag_output = merged[['Dist Name Clean', 'State Name', 'Year', 'Crop', 'rag_text']]
rag_output.to_csv("rag_chunks_final.csv", index=False)

Total rows: 25210
Unique rag_text: 25210
